In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv()
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.7)

True

In [3]:
class JokeState(TypedDict):
    topic : str
    joke : str
    explanation : str

In [4]:
def generate_joke(state: JokeState):
    prompt = f"Generate a joke about {state['topic']}."
    respose = llm.invoke(prompt)

    return {
        "joke": respose.content
        }

In [5]:
def generate_explanation(state: JokeState):
    prompt = f"Explain the joke: {state['joke']}."
    respose = llm.invoke(prompt)

    return {
        "explanation": respose.content
        }

In [ ]:
graph = StateGraph(JokeState)

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

graph.add_edge( START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)

checkpoint=InMemorySaver() # create a checkpoint to save the conversation history in memory, it tells the graph to save the state of the conversation at each step, allowing us to retrieve it later if needed.

workflow = graph.compile(checkpointer=checkpoint)

In [8]:
config1 = {"configurable":{"thread_id": "1"}}
workflow.invoke({"topic": "programming"}, config=config1)

{'topic': 'programming',
 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.',
 'explanation': 'The joke is a play on words, using the multiple meanings of "bugs."\n\nIn computing, a "bug" refers to a defect or error in a program or software. In this sense, the phrase "light attracts bugs" suggests that programmers prefer dark mode because it reduces the likelihood of bugs in their code.\n\nHowever, the phrase "light attracts bugs" is also a literal reference to the fact that insects, such as mosquitoes and flies, are attracted to light sources. This is a common observation in everyday life.\n\nThe joke relies on this double meaning of "bugs" to create a pun, making it a clever and humorous explanation for why programmers might prefer dark mode.'}

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'programming', 'joke': 'Why do programmers prefer dark mode?\n\nBecause light attracts bugs.', 'explanation': 'The joke is a play on words, using the multiple meanings of "bugs."\n\nIn computing, a "bug" refers to a defect or error in a program or software. In this sense, the phrase "light attracts bugs" suggests that programmers prefer dark mode because it reduces the likelihood of bugs in their code.\n\nHowever, the phrase "light attracts bugs" is also a literal reference to the fact that insects, such as mosquitoes and flies, are attracted to light sources. This is a common observation in everyday life.\n\nThe joke relies on this double meaning of "bugs" to create a pun, making it a clever and humorous explanation for why programmers might prefer dark mode.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f14f6dc-5da4-65bb-8002-aa2ab2be6ea1'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created

In [11]:
config2 = {"configurable":{"thread_id": "2"}}
workflow.invoke({"topic": "sports"}, config=config2)

{'topic': 'sports',
 'joke': 'Why did the golfer wear two pairs of pants?\n\nIn case he got a hole in one.',
 'explanation': 'This joke is a play on words. "Hole in one" has a double meaning here. In golf, a "hole in one" refers to when a player hits the ball directly into the hole with one stroke. However, in regular language, "a hole in one" also means a single hole or tear in a piece of clothing, such as a pair of pants.\n\nThe joke relies on this double meaning to create a pun. The golfer wears two pairs of pants "in case he got a hole in one" (the golf term), but the phrase also implies that the golfer is wearing extra pants to avoid getting a hole in one (the clothing term), which is a clever and humorous twist.'}

In [12]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'sports', 'joke': 'Why did the golfer wear two pairs of pants?\n\nIn case he got a hole in one.', 'explanation': 'This joke is a play on words. "Hole in one" has a double meaning here. In golf, a "hole in one" refers to when a player hits the ball directly into the hole with one stroke. However, in regular language, "a hole in one" also means a single hole or tear in a piece of clothing, such as a pair of pants.\n\nThe joke relies on this double meaning to create a pun. The golfer wears two pairs of pants "in case he got a hole in one" (the golf term), but the phrase also implies that the golfer is wearing extra pants to avoid getting a hole in one (the clothing term), which is a clever and humorous twist.'}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f14f6e2-cfa2-6456-8002-1dc5694ab437'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-05-14T08:23:24.478370+00:00', parent_config={